In [1]:
import json
import pandas as pd

WEATHER_FILE = "../data/raw/weather_6158731_2025.json"

with open(WEATHER_FILE, "r", encoding="utf-8") as file:
    data = json.load(file)

weather = pd.DataFrame(
    feature["properties"]
    for feature in data["features"]
)

weather.head()

,STATION_NAME,CLIMATE_IDENTIFIER,ID,LOCAL_DATE,PROVINCE_CODE,LOCAL_YEAR,LOCAL_MONTH,LOCAL_DAY,LOCAL_HOUR,UTC_DATE,...,WINDCHILL,WINDCHILL_FLAG,WIND_DIRECTION,WIND_DIRECTION_FLAG,WIND_SPEED,WIND_SPEED_FLAG,STN_ID,LONGITUDE_DECIMAL_DEGREES,LATITUDE_DECIMAL_DEGREES,FLAG
0,TORONTO INTL A,6158731,6158731.2025.1.1.3,2025-01-01 03:00:00,ON,2025,1,1,3,2025-01-01T08:00:00,...,NaN,None,34.0,None,26.0,None,51459,-79.630556,43.676667,None
1,TORONTO INTL A,6158731,6158731.2025.1.1.0,2025-01-01 00:00:00,ON,2025,1,1,0,2025-01-01T05:00:00,...,NaN,None,35.0,None,27.0,None,51459,-79.630556,43.676667,None
2,TORONTO INTL A,6158731,6158731.2025.1.1.1,2025-01-01 01:00:00,ON,2025,1,1,1,2025-01-01T06:00:00,...,NaN,None,35.0,None,29.0,None,51459,-79.630556,43.676667,None
3,TORONTO INTL A,6158731,6158731.2025.1.1.2,2025-01-01 02:00:00,ON,2025,1,1,2,2025-01-01T07:00:00,...,NaN,None,35.0,None,30.0,None,51459,-79.630556,43.676667,None
4,TORONTO INTL A,6158731,6158731.2025.1.1.4,2025-01-01 04:00:00,ON,2025,1,1,4,2025-01-01T09:00:00,...,NaN,None,34.0,None,23.0,None,51459,-79.630556,43.676667,None


In [2]:
weather = weather[["LOCAL_DATE", "TEMP"]].copy()

In [3]:
weather = weather.rename(
    columns={
        "LOCAL_DATE": "timestamp",
        "TEMP": "temperature"
    }
)

weather["timestamp"] = pd.to_datetime(weather["timestamp"])
weather = weather.sort_values("timestamp").reset_index(drop=True)
weather.head()

,timestamp,temperature
0,2025-01-01 00:00:00,1.6
1,2025-01-01 01:00:00,1.6
2,2025-01-01 02:00:00,1.4
3,2025-01-01 03:00:00,0.8
4,2025-01-01 04:00:00,1.4


In [4]:
weather[weather["temperature"].isna()]

,timestamp,temperature
5196,2025-08-05 12:00:00,NaN


In [5]:
print("Rows:", len(weather))
print("Duplicate timestamps:", weather["timestamp"].duplicated().sum())
print("Missing temperatures:", weather["temperature"].isna().sum())
print("Start:", weather["timestamp"].min())
print("End:", weather["timestamp"].max())

Rows: 8760
Duplicate timestamps: 0
Missing temperatures: 1
Start: 2025-01-01 00:00:00
End: 2025-12-31 23:00:00


In [6]:
weather[
    (weather["timestamp"] >= "2025-08-05 10:00:00") &
    (weather["timestamp"] <= "2025-08-05 14:00:00")
]

,timestamp,temperature
5194,2025-08-05 10:00:00,24.9
5195,2025-08-05 11:00:00,26.0
5196,2025-08-05 12:00:00,NaN
5197,2025-08-05 13:00:00,27.2
5198,2025-08-05 14:00:00,27.9


In [7]:
weather["temperature"] = weather["temperature"].interpolate(
    method="linear"
)

In [8]:
weather[
    (weather["timestamp"] >= "2025-08-05 10:00:00") &
    (weather["timestamp"] <= "2025-08-05 14:00:00")
]

,timestamp,temperature
5194,2025-08-05 10:00:00,24.9
5195,2025-08-05 11:00:00,26.0
5196,2025-08-05 12:00:00,26.6
5197,2025-08-05 13:00:00,27.2
5198,2025-08-05 14:00:00,27.9


In [9]:
weather["temperature"].isna().sum()

np.int64(0)

In [10]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = PROCESSED_DIR / "toronto_weather_2025_clean.csv"
weather.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {len(weather)} rows to {OUTPUT_FILE}")

Saved 8760 rows to ..\data\processed\toronto_weather_2025_clean.csv
